In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

def load_kinker_metadata_txt(txt_path):
    """
    Load a tab-delimited metadata .txt file into a pandas DataFrame.

    Assumes:
    - First row = column names
    - Second row = TYPE metadata (to be skipped)
    - Remaining rows = data

    Parameters
    ----------
    txt_path : str or Path
        Path to the .txt file

    Returns
    -------
    pd.DataFrame
    """
    txt_path = Path(txt_path)

    if not txt_path.exists():
        raise FileNotFoundError(f"File not found: {txt_path}")

    df = pd.read_csv(
        txt_path,
        sep="\t",
        header=0,
        skiprows=[1],   # skip the TYPE row
        na_values=["NA"]
    )

    return df



def build_schmidt_cell_metadata(guide_calls_path, out_path):
    """
    Build cell metadata table from CellRanger guide calls.

    Parameters
    ----------
    guide_calls_path : str
        Path to cellranger-guidecalls-aggregated-unfiltered.txt
    out_path : str
        Output path for cell_metadata.txt
    """

    guide_calls = pd.read_csv(
        guide_calls_path,
        sep="\t",
        na_values=["NA"]
    )

    metadata = (
        guide_calls
        # Filter for guide singlets
        .loc[guide_calls["num_features"] == 1]
        # Ensure num_umis is numeric
        .assign(num_umis=lambda df: pd.to_numeric(df["num_umis"], errors="coerce"))
        # Filter for number of UMIs >= 5
        .loc[lambda df: df["num_umis"] >= 5]
        # Add derived columns
        .assign(
            condition=lambda df: np.where(
                df["cell_barcode"].str.contains(r"-1|-2|-3|-4", regex=True),
                "Nostim",
                "Stim"
            ),
            gene=lambda df: df["feature_call"].str[:-2],  # equivalent to str_sub(end = -3)
            crispr=lambda df: np.where(
                df["feature_call"].str[:-2].str.contains("NO-TARGET"),
                "NT",
                "perturbed"
            )
        )
        # Select and rename columns
        .loc[:, ["cell_barcode", "condition", "crispr", "feature_call", "gene"]]
        .rename(columns={"feature_call": "guide_id"})
    )

    metadata.to_csv(out_path, sep="\t", index=False)

    return metadata


In [3]:
kinker_df = load_kinker_metadata_txt("../data/kinker_SCP542_metadata.txt")

#print(kinker_df.head())
print(kinker_df.columns)

schmidt_df = build_schmidt_cell_metadata(
    "../data/GSE190604_cellranger-guidecalls-aggregated-unfiltered.txt",
    "../data/cell_metadata.txt"
)

print("#######################################################################################")

#print(schmidt_df.head())
print(schmidt_df.columns)


C:\Users\Stephen\AppData\Local\Temp\ipykernel_34448\4013578861.py:28: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Index(['NAME', 'Cell_line', 'Pool_ID', 'Cancer_type', 'Genes_expressed',
       'Discrete_cluster_minpts5_eps1.8', 'Discrete_cluster_minpts5_eps1.5',
       'Discrete_cluster_minpts5_eps1.2', 'CNA_subclone', 'SkinPig_score',
       'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score',
       'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score',
       'ProtDegra_score', 'G1/S_score', 'G2/M_score'],
      dtype='object')
#######################################################################################
Index(['cell_barcode', 'condition', 'crispr', 'guide_id', 'gene'], dtype='object')
